In [5]:
# This script:
# 1. Loads the tuned XGBoost model and test set
# 2. Full evaluation: ROC-AUC, PR-AUC, F1, calibration, threshold analysis
# 3. ROC and Precision-Recall curves
# 4. SHAP explainability: global feature importance + individual prediction explanation
# 5. Partial dependence for top features
# 6. Saves all plots and a summary metrics table

import pandas as pd
import numpy as np
import os
import warnings
import joblib
warnings.filterwarnings("ignore")
 
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    classification_report, confusion_matrix,
    RocCurveDisplay, PrecisionRecallDisplay,
    brier_score_loss)
from sklearn.calibration import calibration_curve
from sklearn.inspection import PartialDependenceDisplay
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import shap

In [6]:
OUTPUT_DIR = r"C:\Users\USER\PycharmProjects\Non-med-dementia-risk\outputs"
 
X_train = np.load(os.path.join(OUTPUT_DIR, "X_train.npy"))
X_test  = np.load(os.path.join(OUTPUT_DIR, "X_test.npy"))
y_train = np.load(os.path.join(OUTPUT_DIR, "y_train.npy"))
y_test  = np.load(os.path.join(OUTPUT_DIR, "y_test.npy"))
 
feature_cols = pd.read_csv(os.path.join(OUTPUT_DIR, "feature_names.csv")).iloc[:, 0].tolist()
model        = joblib.load(os.path.join(OUTPUT_DIR, "xgboost_tuned.pkl"))
 
print(f"X_test: {X_test.shape}")
print(f"Features: {len(feature_cols)}")

X_test: (10508, 44)
Features: 44


In [7]:
# 1. Core metrics

y_prob = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)
 
roc_auc  = roc_auc_score(y_test, y_prob)
pr_auc   = average_precision_score(y_test, y_prob)
f1       = f1_score(y_test, y_pred)
brier    = brier_score_loss(y_test, y_prob)
 
print("=== Core Metrics (XGBoost tuned, test set) ===")
print(f"ROC-AUC:     {roc_auc:.4f}")
print(f"PR-AUC:      {pr_auc:.4f}")
print(f"F1:          {f1:.4f}")
print(f"Brier Score: {brier:.4f}  (lower is better; 0=perfect, 0.25=random)")
print()
print(classification_report(y_test, y_pred, target_names=["Not at risk", "At risk"]))
 

=== Core Metrics (XGBoost tuned, test set) ===
ROC-AUC:     0.7306
PR-AUC:      0.7656
F1:          0.7152
Brier Score: 0.2081  (lower is better; 0=perfect, 0.25=random)

              precision    recall  f1-score   support

 Not at risk       0.66      0.59      0.62      4757
     At risk       0.69      0.75      0.72      5751

    accuracy                           0.67     10508
   macro avg       0.67      0.67      0.67     10508
weighted avg       0.67      0.67      0.67     10508



In [8]:
# 2. Threshold analysis
#
# Default threshold of 0.5 is not always optimal. This section shows how
# precision, recall, and F1 vary across thresholds so we can discuss
# the trade-off explicitly in the report.

from sklearn.metrics import precision_recall_curve
 
precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob)
f1_scores = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-8)
best_thresh_idx = np.argmax(f1_scores)
best_threshold  = thresholds[best_thresh_idx]
 
print(f"Optimal threshold (max F1): {best_threshold:.3f}")
print(f"  Precision at optimal:     {precisions[best_thresh_idx]:.4f}")
print(f"  Recall at optimal:        {recalls[best_thresh_idx]:.4f}")
print(f"  F1 at optimal:            {f1_scores[best_thresh_idx]:.4f}")
 
# Apply optimal threshold
y_pred_opt = (y_prob >= best_threshold).astype(int)
print(f"\nClassification report at optimal threshold ({best_threshold:.3f}):")
print(classification_report(y_test, y_pred_opt, target_names=["Not at risk", "At risk"]))
 


Optimal threshold (max F1): 0.397
  Precision at optimal:     0.6409
  Recall at optimal:        0.8668
  F1 at optimal:            0.7369

Classification report at optimal threshold (0.397):
              precision    recall  f1-score   support

 Not at risk       0.72      0.41      0.52      4757
     At risk       0.64      0.87      0.74      5751

    accuracy                           0.66     10508
   macro avg       0.68      0.64      0.63     10508
weighted avg       0.68      0.66      0.64     10508



In [9]:
# 3. Plots — ROC, PR curve, Calibration, Confusion Matrix

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
 
# ROC curve
RocCurveDisplay.from_predictions(
    y_test, y_prob, name=f"XGBoost (AUC={roc_auc:.3f})", ax=axes[0]
)
axes[0].plot([0, 1], [0, 1], "k--", label="Random")
axes[0].set_title("ROC Curve")
axes[0].legend(loc="lower right")
 
# PR curve
PrecisionRecallDisplay.from_predictions(
    y_test, y_prob, name=f"XGBoost (AP={pr_auc:.3f})", ax=axes[1]
)
axes[1].set_title("Precision-Recall Curve")
 
# Calibration curve
prob_true, prob_pred = (y_test, y_prob, n_bins=10)
axes[2].plot(prob_pred, prob_true, "s-", label="XGBoost")
axes[2].plot([0, 1], [0, 1], "k--", label="Perfect calibration")
axes[2].set_xlabel("Mean predicted probability")
axes[2].set_ylabel("Fraction of positives")
axes[2].set_title("Calibration Curve")
axes[2].legend()
 
# Confusion matrix (optimal threshold)
cm = confusion_matrix(y_test, y_pred_opt)
im = axes[3].imshow(cm, interpolation="nearest", cmap="Blues")
axes[3].set_title(f"Confusion Matrix\n(threshold={best_threshold:.3f})")
axes[3].set_xlabel("Predicted"); axes[3].set_ylabel("Actual")
axes[3].set_xticks([0, 1]); axes[3].set_yticks([0, 1])
axes[3].set_xticklabels(["Not at risk", "At risk"])
axes[3].set_yticklabels(["Not at risk", "At risk"])
for i in range(2):
    for j in range(2):
        axes[3].text(j, i, cm[i, j], ha="center", va="center",
                     color="white" if cm[i, j] > cm.max()/2 else "black", fontsize=12)
 
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "step6_evaluation_plots.png"), dpi=150)
print("Saved: step6_evaluation_plots.png")

Saved: step6_evaluation_plots.png


In [10]:
# 4. SHAP — Global Feature Importance
#
# SHAP (SHapley Additive exPlanations) assigns each feature a contribution
# value for each prediction. Unlike standard feature importances, SHAP values
# are theoretically grounded (based on game theory) and show both the magnitude
# AND direction of each feature's effect.
#
# We use the TreeExplainer which is exact and fast for XGBoost.
 
print("\nComputing SHAP values (this may take ~1 minute) ...")
explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
 
# SHAP summary plot — beeswarm (shows distribution + direction)
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values, X_test,
    feature_names=feature_cols,
    show=False,
    max_display=20
)
plt.title("SHAP Summary Plot — Top 20 Features")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "step6_shap_summary.png"), dpi=150, bbox_inches="tight")
print("Saved: step6_shap_summary.png")

# SHAP bar plot — mean absolute SHAP (overall importance)
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values, X_test,
    feature_names=feature_cols,
    plot_type="bar",
    show=False,
    max_display=20
)
plt.title("SHAP Feature Importance — Mean |SHAP value|")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "step6_shap_importance.png"), dpi=150, bbox_inches="tight")
print("Saved: step6_shap_importance.png")


Computing SHAP values (this may take ~1 minute) ...
Saved: step6_shap_summary.png
Saved: step6_shap_importance.png


In [11]:
# 5. Top features — mean absolute SHAP ranking

mean_abs_shap = pd.Series(
    np.abs(shap_values).mean(axis=0),
    index=feature_cols
).sort_values(ascending=False)
 
print("\nTop 15 features by mean |SHAP value|:")
print(mean_abs_shap.head(15).round(4).to_string())
mean_abs_shap.to_csv(os.path.join(OUTPUT_DIR, "step6_shap_ranking.csv"), header=["mean_abs_shap"])


Top 15 features by mean |SHAP value|:
INCALLS         0.3030
EDUC            0.2431
SEX             0.1962
NACCAGEB        0.1756
BIRTHYR         0.1584
INVISITS        0.1307
PACK_YEARS      0.1035
INRELTO_5.0     0.1005
INRELTO_2.0     0.0747
RACE_2.0        0.0670
INLIVWTH        0.0528
RESIDENC_3.0    0.0436
RESIDENC_4.0    0.0424
TOBAC100        0.0417
NACCLIVS_3.0    0.0409


In [12]:
# 6. SHAP dependence plots — top 3 features
#
# Dependence plots show how a single feature's value affects the model output
# (SHAP value), with colour showing interaction with the most correlated feature.
 

top3 = mean_abs_shap.head(3).index.tolist()
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
 
for ax, feat in zip(axes, top3):
    feat_idx = feature_cols.index(feat)
    shap.dependence_plot(
        feat_idx, shap_values, X_test,
        feature_names=feature_cols,
        ax=ax, show=False
    )
    ax.set_title(f"SHAP Dependence: {feat}")
 
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "step6_shap_dependence.png"), dpi=150, bbox_inches="tight")
print(f"Saved: step6_shap_dependence.png  (features: {top3})")

Saved: step6_shap_dependence.png  (features: ['INCALLS', 'EDUC', 'SEX'])


In [13]:
# 7. Summary metrics table — save for report
 
metrics_summary = pd.DataFrame([{
    "Model":          "XGBoost (tuned)",
    "ROC-AUC":        round(roc_auc, 4),
    "PR-AUC":         round(pr_auc, 4),
    "F1 (0.5)":       round(f1, 4),
    "F1 (optimal)":   round(f1_scores[best_thresh_idx], 4),
    "Optimal thresh": round(best_threshold, 3),
    "Brier Score":    round(brier, 4),
}])
 
metrics_summary.to_csv(os.path.join(OUTPUT_DIR, "step6_metrics_summary.csv"), index=False)
print(f"\nSaved: step6_metrics_summary.csv")
print(metrics_summary.to_string(index=False))


Saved: step6_metrics_summary.csv
          Model  ROC-AUC  PR-AUC  F1 (0.5)  F1 (optimal)  Optimal thresh  Brier Score
XGBoost (tuned)   0.7306  0.7656    0.7152        0.7369           0.397       0.2081


In [ ]:
# 8. Full evaluation of all three tuned models
#
# Evaluate tuned Logistic Regression and Random Forest on the same test set
# for a complete model comparison table in the report.
 
all_models = {
    "XGBoost":              joblib.load(os.path.join(OUTPUT_DIR, "xgboost_tuned.pkl")),
    "Random Forest":        joblib.load(os.path.join(OUTPUT_DIR, "random_forest_tuned.pkl")),
    "Logistic Regression":  joblib.load(os.path.join(OUTPUT_DIR, "logistic_regression_tuned.pkl")),
}
 
all_metrics = []
all_probs   = {}

for name, mdl in all_models.items():
    yp      = mdl.predict_proba(X_test)[:, 1]
    yd      = mdl.predict(X_test)
    all_probs[name] = yp
 
    # Optimal threshold per model
    prec, rec, thresh = precision_recall_curve(y_test, yp)
    f1s  = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-8)
    best = np.argmax(f1s)
 
    all_metrics.append({
        "Model":              name,
        "ROC-AUC":            round(roc_auc_score(y_test, yp), 4),
        "PR-AUC":             round(average_precision_score(y_test, yp), 4),
        "F1 (0.5)":           round(f1_score(y_test, yd), 4),
        "F1 (optimal)":       round(f1s[best], 4),
        "Optimal threshold":  round(thresh[best], 3),
        "Brier Score":        round(brier_score_loss(y_test, yp), 4),
    })
    
comparison_all = pd.DataFrame(all_metrics).set_index("Model")
print("\n=== Full Tuned Model Comparison ===")
print(comparison_all.to_string())
comparison_all.to_csv(os.path.join(OUTPUT_DIR, "step6_all_models_comparison.csv"))
print("Saved: step6_all_models_comparison.csv")

In [ ]:
# 9. ROC + Calibration curves — all three tuned models

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
 
# ROC curves
for name, yp in all_probs.items():
    auc = roc_auc_score(y_test, yp)
    RocCurveDisplay.from_predictions(y_test, yp, name=f"{name} (AUC={auc:.3f})", ax=axes[0])
axes[0].plot([0, 1], [0, 1], "k--", label="Random")
axes[0].set_title("ROC Curves — All Tuned Models")
axes[0].legend(loc="lower right", fontsize=8)
 
# Calibration curves
for name, yp in all_probs.items():
    prob_true, prob_pred = calibration_curve(y_test, yp, n_bins=10)
    axes[1].plot(prob_pred, prob_true, "s-", label=name)
axes[1].plot([0, 1], [0, 1], "k--", label="Perfect calibration")
axes[1].set_xlabel("Mean predicted probability")
axes[1].set_ylabel("Fraction of positives")
axes[1].set_title("Calibration Curves — All Tuned Models")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "step6_all_models_roc_calibration.png"), dpi=150)
print("Saved: step6_all_models_roc_calibration.png")